# Question 7


# Unity Catalog Governance, Data Classification & Audit Model

## 1. Sensitive Data Classification (PII Matrix)

For Services and Salesforce implementation environments, sensitive attributes are classified and isolated at scale following a structured governance framework.

| **Table Name**         | **Sensitive Columns (PII / Financial)**                   | **Sensitivity Level**    | **Masking / Restriction Strategy**            |
| ---------------------- | --------------------------------------------------------- | ------------------------ | --------------------------------------------- |
| **`sales.customers`**  | `email`, `phone_number`, `billing_address`, `tax_id`      | High (PII)               | Dynamic Data Masking (`email` / `phone`)      |
| **`hr.employees`**     | `ssn`, `base_salary`, `bank_account_no`, `personal_email` | Critical (PII/Financial) | Column-Level Security (Strictly Restricted)   |
| **`crm.leads`**        | `first_name`, `last_name`, `email_address`, `phone`       | Medium (PII)             | Accessible only to Go-To-Market (GTM) teams   |
| **`finance.invoices`** | `payment_details`, `credit_card_mask`, `billing_name`     | High (Financial PII)     | Row-Level Filtering by Region / Business Unit |
| **`support.tickets`**  | `customer_notes`, `contact_email`                         | Low-Medium               | Text Redaction Functions on `customer_notes`  |

---

## 2. Unity Catalog Groups & RBAC Matrix

Access control should be managed through **Databricks Unity Catalog groups** rather than individual user-level permissions.

### UC Groups Taxonomy

* `uc_admin_secops`: Security and Platform Governance Administrators.
* `uc_group_data_engineers`: ETL/ELT pipelines and pipeline execution service principals.
* `uc_group_data_analysts`: Business operations, reporting, and BI users.
* `uc_group_hr_execs`: HR Management and Payroll users.
* `uc_group_gtm_sales`: Sales performance and Lead processing users.

### Privilege Matrix

| **Group Name**                | **Catalog / Schema Level Access**           | **Table Access Level** | **Sensitive Column Access**             |
| ----------------------------- | ------------------------------------------- | ---------------------- | --------------------------------------- |
| **`uc_admin_secops`**         | `ALL PRIVILEGES` on Metastore               | Full Access            | Full Access (Manages Masking Functions) |
| **`uc_group_data_engineers`** | `USE CATALOG`, `USE SCHEMA`, `CREATE TABLE` | `SELECT`, `MODIFY`     | Masked / Pseudonymized data only        |
| **`uc_group_data_analysts`**  | `USE CATALOG`, `USE SCHEMA`                 | `SELECT`               | Masked Columns (No Raw PII)             |
| **`uc_group_hr_execs`**       | `USE CATALOG (hr)`, `USE SCHEMA (hr)`       | `SELECT` on `hr.*`     | Unmasked HR Data                        |
| **`uc_group_gtm_sales`**      | `USE CATALOG (crm)`, `USE SCHEMA (crm)`     | `SELECT` on `crm.*`    | Raw Lead/Customer Data                  |

---

## 3. Data Masking & Dynamic Column Security Implementation

Dynamic Data Masking and Row-Level Filtering can be implemented in Unity Catalog using SQL masking functions.

### Dynamic Email Masking Function

```sql
-- Create a Dynamic Masking Function for Email
CREATE OR REPLACE FUNCTION main.default.email_mask(email STRING)
RETURN CASE
  WHEN IS_ACCOUNT_GROUP_MEMBER('uc_group_gtm_sales') THEN email
  WHEN IS_ACCOUNT_GROUP_MEMBER('uc_admin_secops') THEN email
  ELSE CONCAT(
    SUBSTR(email, 1, 2),
    '****@',
    SPLIT_PART(email, '@', 2)
  )
END;
```

### Apply Masking Function to Customer Table

```sql
ALTER TABLE sales.customers
ALTER COLUMN email SET MASK main.default.email_mask;
```

### Masking Logic

| **User Group**       | **Email Visibility** |
| -------------------- | -------------------- |
| `uc_group_gtm_sales` | Raw Email            |
| `uc_admin_secops`    | Raw Email            |
| Other Groups         | Masked Email         |

---

## 4. Continuous Compliance Protocol

### 4.1 Automated SIEM Integration

Databricks System Audit Logs can be integrated with external SIEM platforms such as Splunk or Datadog through Webhooks, Log Analytics, or other supported log delivery mechanisms.

**Objectives:**

* Detect suspicious bulk data downloads.
* Identify unauthorized data access.
* Monitor high-volume queries against sensitive tables.
* Trigger automated alerts for security incidents.

### 4.2 Quarterly Access Certification

Generate an active permissions matrix dynamically and conduct a quarterly access review with business owners and data owners.

The review should include:

* User-to-group membership.
* Group-to-catalog permissions.
* Group-to-schema permissions.
* Table-level privileges.
* Sensitive-data access.
* Masking exceptions.

### 4.3 Automated Catalog Tagging

Sensitive columns should be classified using Unity Catalog tags.

Example:

```text
PII = True
Classification = Restricted
```

### Example Classification

| **Tag**            | **Value**    |
| ------------------ | ------------ |
| `PII`              | `True`       |
| `Classification`   | `Restricted` |
| `Data_Domain`      | `Customer`   |
| `Masking_Required` | `True`       |

Automated governance scripts can use these tags to identify whether:

* Sensitive columns have been properly classified.
* Required masking is missing.
* Appropriate access policies exist.
* Governance standards are consistently applied across the platform.

---

## 5. Governance Control Summary

The overall governance model follows a **Least Privilege + Defense-in-Depth** approach:

```text
Data Classification
        ↓
Unity Catalog Tags
        ↓
Groups & RBAC
        ↓
Table / Column / Row-Level Security
        ↓
Dynamic Data Masking
        ↓
Audit Logs
        ↓
SIEM Monitoring
        ↓
Quarterly Access Certification
        ↓
Continuous Compliance
```

### Key Principles

* **Least Privilege:** Users should receive only the permissions required to perform their responsibilities.
* **Group-Based Access:** Direct user-level grants should be minimized in favor of group-based access management.
* **Sensitive Data Isolation:** PII and financial data should be placed under restricted access controls.
* **Dynamic Masking:** Sensitive values should automatically be masked for unauthorized users.
* **Row-Level Security:** Records can be restricted based on business unit, region, organizational boundary, or other access attributes.
* **Auditability:** Access to sensitive data should be continuously logged and reviewed.
* **Automated Compliance:** Governance checks can be automated using tags, audit logs, and permission matrices.
* **Periodic Certification:** Access permissions should be reviewed and certified by appropriate business and data owners on a quarterly basis.


# Question 8
Extend the SCD Type 2 pattern to track changes across 3+ columns simultaneously, and handle the
edge case of a customer record that hasn't changed since the last load (it should not create a false
new version).


In [0]:
from pyspark.sql.functions import *

# 1. Source Path (Jahan tumhari Raw CSV files aayengi)
raw_source_path = "/Volumes/cyntexa_dev/day_8/my_volume/customers/"

# 2. Schema Checkpointing Path (Auto Loader state track karne ke liye)
checkpoint_path = "/Volumes/cyntexa_dev/day_8/my_volume/bronze_customers/"
schema_path = "/Volumes/cyntexa_dev/day_8/my_volume/schema_location/"
# 3. Auto Loader Read Stream (Zero Cleaning - Pure Raw Data Capture)
raw_stream_df = spark.readStream \
    .format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("cloudFiles.schemaLocation",schema_path) \
    .option("header", "true") \
    .load(raw_source_path) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_file_name", col("_metadata.file_name")) \
    .withColumn("source_file_path", col("_metadata.file_path"))

# 4. Write Stream to Bronze Table (Trigger Once taaki ek batch me run ho kar rukk jaaye)
query = raw_stream_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .trigger(availableNow=True) \
    .toTable("cyntexa_dev.day_8.bronze_customers")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS cyntexa_dev.day_8.silver_customers_scd2 (
    customer_id STRING,
    first_name STRING,
    last_name STRING,
    email STRING,
    city STRING,
    signup_date STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP,
    is_current BOOLEAN
) USING DELTA;

In [0]:
from pyspark.sql.functions import col, trim, lower, row_number, to_timestamp
from pyspark.sql.window import Window

# Bronze read
bronze_df = spark.read.table("cyntexa_dev.day_8.bronze_customers")

# Basic Cleaning (Null Check + String Trimming)
cleaned_df = bronze_df \
    .withColumn("customer_id", trim(col("customer_id"))) \
    .withColumn("first_name", trim(col("first_name"))) \
    .withColumn("last_name", trim(col("last_name"))) \
    .withColumn("email", lower(trim(col("email")))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("signup_date", trim(col("signup_date"))) \
    .withColumn("updated_at", to_timestamp(col("updated_at"))) \
    .filter(col("customer_id").isNotNull() & (col("customer_id") != "NULL") & (col("customer_id") != "")) \
    .filter(col("updated_at").isNotNull())

# Batch Level Deduplication (Latest updated_at standard pick karega)
window_spec = Window.partitionBy("customer_id").orderBy(col("updated_at").desc())

dedup_cleaned_df = cleaned_df \
    .withColumn("row_num", row_number().over(window_spec)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# Temp View for MERGE
dedup_cleaned_df.createOrReplaceTempView("dedup_cleaned_df_customers")

In [0]:
%sql
--  Step 1 ----------------------
merge into  cyntexa_dev.day_8.silver_customers_scd2  t
using dedup_cleaned_df_customers source
on t.customer_id = source.customer_id  and t.end_date is null
when matched and (
    t.email <> source.email or 
    t.city <> source.city or
    t.first_name <> source.first_name or 
    t.last_name <> source.last_name
     )  then update 
     set 
t.end_date = source.updated_at,
t.is_current = false

WHEN NOT MATCHED THEN
  INSERT (
    customer_id, 
    first_name, 
    last_name, 
    email, 
    city, 
    signup_date, 
    start_date, 
    end_date, 
    is_current
  )
  VALUES (
    source.customer_id, 
    source.first_name, 
    source.last_name, 
    source.email, 
    source.city, 
    source.signup_date, 
    source.updated_at, 
    NULL, 
    true
  );

-- step 2---------------
INSERT INTO cyntexa_dev.day_8.silver_customers_scd2  (
  customer_id, first_name, last_name, email, city, signup_date, start_date, end_date, is_current
)
select s.customer_id , s.first_name, s.last_name, s.email, s.city, s.signup_date , s.updated_at, null, true from dedup_cleaned_df_customers  s   
left join  cyntexa_dev.day_8.silver_customers_scd2 t
on s.customer_id = t.customer_id and t.end_date is null  
where t.customer_id is null   


# Question 9

(Data Analyst) Using the SCD Type 2 history table, build a customer retention/churn-over-time report
that depends on point-in-time correctness, and explain why a simple 'current state' table would give
the wrong answer here.

In [0]:
%sql

WITH month_ends AS (
    SELECT DISTINCT
        LAST_DAY(start_date) AS month_end_date
    FROM cyntexa_dev.day_8.silver_customers_scd2
),

active_customers AS (
    SELECT
        m.month_end_date,
        c.city,
        c.customer_id
    FROM month_ends m
    JOIN cyntexa_dev.day_8.silver_customers_scd2 c
        ON c.start_date <= m.month_end_date
       AND (c.end_date IS NULL OR c.end_date > m.month_end_date)
),

monthly_summary AS (
    SELECT
        month_end_date,
        city,
        COUNT(DISTINCT customer_id) AS active_customers
    FROM active_customers
    GROUP BY month_end_date, city
),

previous_month AS (
    SELECT
        month_end_date,
        city,
        active_customers,
        LAG(active_customers) OVER (
            PARTITION BY city
            ORDER BY month_end_date
        ) AS previous_month_active_customers
    FROM monthly_summary
)

SELECT
    month_end_date,
    city,
    active_customers,
    previous_month_active_customers,
    GREATEST(
        previous_month_active_customers - active_customers,
        0
    ) AS churned_customers,
    ROUND(
        GREATEST(
            previous_month_active_customers - active_customers,
            0
        ) / previous_month_active_customers * 100,
        2
    ) AS churn_rate
FROM previous_month
ORDER BY month_end_date, city;  


Because retention/churn is a historical, point-in-time analysis. We need to know the customer's state at each past month-end, not just their state today. An SCD Type 2 table preserves historical versions using start_date and end_date, allowing us to reconstruct the correct state for any point in time. A current-state table only contains the latest state, so historical months can be incorrectly attributed to the customer's current state.